# 04 · Estimar o preço de um apartamento

Este notebook usa o modelo treinado no notebook 03 para **estimar o preço de qualquer apartamento em Lisboa**. Basta descrever a casa e correr as células.

| | |
|---|---|
| **Entrada** | `models/modelo_final.joblib` (criado pelo notebook 03) |
| **Saída** | estimativa de preço com intervalo provável |

> Lembrete: o texto explica *o quê* e *porquê*; os comentários (`#`) dentro do código explicam *como*.

> ⚠️ O modelo estima o **preço pedido em anúncios**, não o preço final de escritura. Foi treinado com apartamentos até **2,5 M€**.

## 0. Carregar o modelo

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

# Procurar a pasta do projeto: começamos na pasta atual (Path.cwd()) e subimos
# pasta a pasta (.parent) até encontrar a pasta 'src'. Assim o notebook funciona
# quer seja aberto a partir da pasta 'notebooks/', quer a partir da raiz do projeto.
RAIZ_PROJETO = Path.cwd()
while not (RAIZ_PROJETO / "src").is_dir() and RAIZ_PROJETO != RAIZ_PROJETO.parent:
    RAIZ_PROJETO = RAIZ_PROJETO.parent
sys.path.append(str(RAIZ_PROJETO))   # agora o Python consegue encontrar o módulo 'src'
from src.utils import FICHEIRO_MODELO, preparar_variaveis, formatar_euros, num_pt

# joblib.load abre o ficheiro gravado no notebook 03: é um dicionário com o modelo
# treinado e a informação de contexto (variáveis, margem de erro, limite de preço)
pacote = joblib.load(FICHEIRO_MODELO)
modelo = pacote["modelo"]

# "Desempacotar" o par (inferior, superior) em duas variáveis
limite_inferior, limite_superior = pacote["margem"]

print(f"Modelo: {pacote['nome']} | treinado com {pacote['n_casas_treino']} apartamentos "
      f"até {num_pt(pacote['preco_max']/1e6)} M€")
print(f"Margem (80% dos casos): {limite_inferior:.0%} a {limite_superior:.0%} da estimativa")

Modelo: XGBoost | treinado com 1553 apartamentos até 2,5 M€
Margem (80% dos casos): 80% a 127% da estimativa


## 1. A função de estimativa

Primeiro verificamos que valores o modelo aceita para a freguesia, o estado e o tipo — se escrevermos uma freguesia com um erro, o modelo avisa em vez de dar uma estimativa errada em silêncio.

Depois, a função `estimar_preco` recebe a descrição de uma casa (um dicionário Python) e devolve:
- a **estimativa** do preço,
- um **intervalo provável** (onde ficou o preço real em 80% das casas testadas),
- o **preço por m²** correspondente,
- um **aviso** se a casa estiver perto do limite de 2,5 M€ (fora do âmbito do modelo).

In [2]:
# As categorias que o modelo conhece (aprendidas no treino).
# Se escrevermos uma freguesia com um erro (ex.: "Aroios"), o modelo não dá erro
# sozinho — simplesmente ignora a freguesia e a estimativa sai errada sem aviso.
# Por isso verificamos os valores antes de prever.
codificador = modelo[0].named_transformers_["categorias"]   # o OneHotEncoder do pipeline
VALORES_ACEITES = {nome: list(valores) for nome, valores in
                   zip(["freguesia", "estado", "tipo"], codificador.categories_)}


def prever(casa):
    '''Devolve a estimativa do modelo, em euros, para uma casa (dicionário).'''
    # 0) Verificar se as categorias existem no treino
    for campo, aceites in VALORES_ACEITES.items():
        if casa[campo] not in aceites:
            raise ValueError(f"'{casa[campo]}' não é um valor válido para '{campo}'. "
                             f"Valores aceites: {', '.join(aceites)}")

    # 1) Transformar o dicionário numa tabela com uma linha (o modelo espera uma tabela)
    #    e acrescentar as variáveis derivadas (log_area, etc.) EXATAMENTE como no treino
    dados_casa = preparar_variaveis(pd.DataFrame([casa]))

    # 2) Ficar com as colunas pela mesma ordem usada no treino
    dados_casa = dados_casa[pacote["variaveis"]]

    # 3) Garantir que as colunas numéricas são números decimais (None → vazio/NaN)
    dados_casa = dados_casa.astype({c: float for c in pacote["variaveis"] if c not in VALORES_ACEITES})

    # 4) Prever. O modelo devolve o LOGARITMO do preço → np.exp converte para euros.
    #    [0] porque .predict devolve uma lista (uma previsão por linha; aqui só há uma)
    return float(np.exp(modelo.predict(dados_casa))[0])


def estimar_preco(casa):
    '''Estimativa "apresentável": valor, intervalo provável, preço por m² e avisos.'''
    estimativa = prever(casa)

    # round(x, -3) arredonda aos milhares
    resultado = {
        "Estimativa": formatar_euros(round(estimativa, -3)),
        "Intervalo provável (80%)": f"{formatar_euros(round(estimativa * limite_inferior, -3))} – "
                                    f"{formatar_euros(round(estimativa * limite_superior, -3))}",
        "Preço por m²": formatar_euros(estimativa / casa["area_m2"]),
    }
    # Aviso se a casa estiver perto ou acima do limite de preço do modelo
    if estimativa * limite_superior > pacote["preco_max"]:
        resultado["⚠️ Aviso"] = ("Casa perto ou acima do limite do modelo "
                                 f"({num_pt(pacote['preco_max']/1e6)} M€): estimativa pouco fiável.")
    return pd.Series(resultado, name="Resultado")

## 2. Descrever o apartamento

Preenche as características abaixo. Se não souberes algum valor, escreve `None` — o modelo lida com isso.

| Campo | Valores possíveis |
|---|---|
| `freguesia` | nome da freguesia de Lisboa (ex.: `"Arroios"`, `"Estrela"`, `"Benfica"`) |
| `latitude`, `longitude` | coordenadas — no Google Maps, clica com o botão direito no local e copia os números |
| `estado` | `"good"` (bom estado), `"renew"` (para recuperar), `"newdevelopment"` (empreendimento novo) |
| `tipo` | `"flat"` (apartamento), `"duplex"`, `"penthouse"` (cobertura), `"studio"` (estúdio) |
| `andar` | número (0 = rés-do-chão) |
| sim/não | `1` = sim, `0` = não, `None` = não sei |

### Exemplo 1 — T2 em Arroios

In [3]:
# Um dicionário Python: cada linha é "característica": valor
t2_arroios = {
    "freguesia": "Arroios",
    "latitude": 38.7280,
    "longitude": -9.1350,
    "area_m2": 90,
    "quartos": 2,
    "casas_banho": 2,
    "andar": 3,
    "estado": "good",
    "tipo": "flat",
    "elevador": 1,
    "empreendimento_novo": 0,
    "terraco": 0,
    "piscina": 0,
    "jardim": 0,
    "ar_condicionado": 1,
    "arrecadacao": None,
    "estacionamento": 0,
}
estimar_preco(t2_arroios).to_frame()

,Resultado
Estimativa,592 000 €
Intervalo provável (80%),475 000 € – 749 000 €
Preço por m²,6 574 €


### Exemplo 2 — T1 para recuperar em Penha de França

In [4]:
t1_penha = {
    "freguesia": "Penha de França",
    "latitude": 38.7270,
    "longitude": -9.1260,
    "area_m2": 55,
    "quartos": 1,
    "casas_banho": 1,
    "andar": 2,
    "estado": "renew",
    "tipo": "flat",
    "elevador": 0,
    "empreendimento_novo": 0,
    "terraco": None,
    "piscina": None,
    "jardim": None,
    "ar_condicionado": 0,
    "arrecadacao": None,
    "estacionamento": 0,
}
estimar_preco(t1_penha).to_frame()

,Resultado
Estimativa,295 000 €
Intervalo provável (80%),237 000 € – 373 000 €
Preço por m²,5 359 €


## 3. Experiência: quanto vale cada característica?

Uma forma interessante de "interrogar" o modelo é mudar **uma só característica** e ver como a estimativa reage. Partimos do T2 em Arroios e alteramos uma coisa de cada vez.

(Lembra-te: isto mostra o que o **modelo aprendeu** com os anúncios, não uma verdade absoluta sobre o mercado.)

In [5]:
variantes = {
    "T2 em Arroios (original)": {},
    "+ estacionamento": {"estacionamento": 1},
    "+ 1 casa de banho (3 no total)": {"casas_banho": 3},
    "empreendimento novo": {"estado": "newdevelopment", "empreendimento_novo": 1},
    "sem elevador": {"elevador": 0},
    "+ 20 m² (110 m²)": {"area_m2": 110},
}

linhas = []
for nome, alteracoes in variantes.items():
    # {**a, **b} junta dois dicionários; quando uma chave se repete, ganha o valor de b.
    # Ou seja: "a casa original, mas com estas alterações".
    casa = {**t2_arroios, **alteracoes}
    linhas.append({"cenário": nome, "estimativa": prever(casa)})

experiencia = pd.DataFrame(linhas).set_index("cenário")
base = experiencia["estimativa"].iloc[0]
# Diferença em % face à casa original (a primeira linha)
experiencia["diferença"] = experiencia["estimativa"] / base - 1
experiencia.style.format({"estimativa": lambda v: formatar_euros(round(v, -3)), "diferença": "{:+.0%}"})

,estimativa,diferença
cenário,,
T2 em Arroios (original),592 000 €,+0%
+ estacionamento,709 000 €,+20%
+ 1 casa de banho (3 no total),610 000 €,+3%
empreendimento novo,696 000 €,+18%
sem elevador,523 000 €,-12%
+ 20 m² (110 m²),638 000 €,+8%


## Notas finais

- A estimativa é tão boa quanto os dados: em freguesias com **poucos anúncios** (ex.: Carnide, Ajuda, Santa Clara) a incerteza é maior.
- O intervalo de 80% significa que, em **1 de cada 5 casas**, o preço real fica **fora** desse intervalo.
- Para melhorar o modelo, o mais eficaz é **juntar mais exportações** em `data/raw/` e voltar a correr os notebooks 01 → 03.